In [0]:
CATALOG =  "dev_farm"
BRZ_SCHEMA = "brz_sensor"
SLV_SCHEMA = "slv_sensor"
GLD_SCHEMA = "gld_sensor"
MEASURE_TABLE = "cow_measurements"

#Data Model creation

### Schemas

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRZ_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SLV_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GLD_SCHEMA}")

### Tables

ZeroBus target table <b>cow_measurements</b>

In [0]:
spark.sql(f'''
            CREATE OR REPLACE TABLE {CATALOG}.{BRZ_SCHEMA}.{MEASURE_TABLE} (
            id INTEGER,
            time_serie_event LONG,
            payload variant,
            CONSTRAINT cow_measurements_pk PRIMARY KEY (time_serie_event TIMESERIES) 
            )
            '''
)

###Volume

In [0]:

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRZ_SCHEMA}.resources");

## CowSensor metrics materializaed view

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW dev_farm.gld_sensor.cow_lastest_sensor_data_view
AS
SELECT 
  c.*,
  to_date(to_timestamp(from_unixtime(mp.time_serie_event)), 'yyyy-MM-dd-') AS milk_time_serie_event,
  mp.sensor_id AS milk_sensor_id,
  mp.value AS milk_value,
  to_date(to_timestamp(from_unixtime(w.time_serie_event)), 'yyyy-MM-dd')
   AS weight_time_serie_event,
  w.sensor_id AS weight_sensor_id,
  w.value AS weight_value
FROM `farm-olpt`.public.cows c
JOIN (
  SELECT cow_id, 
         MAX(time_serie_event) AS max_time_serie_event
  FROM dev_farm.slv_sensor.cow_milk_production
  GROUP BY cow_id
) mp_latest
  ON c.id = mp_latest.cow_id
LEFT JOIN dev_farm.slv_sensor.cow_milk_production mp
  ON c.id = mp.cow_id AND mp.time_serie_event = mp_latest.max_time_serie_event
JOIN (
  SELECT cow_id, 
         MAX(time_serie_event) AS max_time_serie_event
  FROM dev_farm.slv_sensor.cow_weight
  GROUP BY cow_id
) w_latest
  ON c.id = w_latest.cow_id
LEFT JOIN dev_farm.slv_sensor.cow_weight w
  ON c.id = w.cow_id AND w.time_serie_event = w_latest.max_time_serie_event
ORDER BY c.id


In [0]:
query = """

"""

df = spark.sql(query)
display(df)


